# Data ConsolidationThis notebook consolidates data from all live databases into a single unified data source.**Purpose:**- Auto-discover all databases in `data/live_databases/`- Extract trades, orders, executors, and controller configs- Combine into consolidated parquet files for efficient analysis- Save to `data/data_sources/` for use in performance reports**When to run:**- After fetching databases from the server (using `fetch_live_databases.ipynb`)- Before running performance analysis- When you want to refresh the consolidated data**Output:**- `consolidated_trades.parquet` - All trades from all bots- `consolidated_orders.parquet` - All orders- `consolidated_executors.parquet` - All executors- `consolidated_controllers.parquet` - All controller configs

In [1]:
from research_notebooks.brigado_v2.data_consolidator import DataConsolidator
from research_notebooks.brigado_v2.file_manager import FileManager
import pandas as pd

# Set pandas display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [2]:
# Initialize consolidator and file manager
consolidator = DataConsolidator()
file_manager = FileManager()

# Discover available databases
databases = consolidator.discover_databases()

print(f"\n{'='*80}")
print("DISCOVERED DATABASES")
print(f"{'='*80}\n")

if databases:
    for i, db in enumerate(databases, 1):
        print(f"[{i}] Bot: {db['bot_name']}")
        print(f"    Database: {db['db_path'].name}")
        print(f"    Size: {db['size_mb']:.2f} MB")
        print(f"    Modified: {db['modified']}")
        print(f"    Config Files: {len(db['config_files'])}")
        if db['config_files']:
            for cfg in db['config_files']:
                print(f"      - {cfg.name}")
        print()
else:
    print("⚠️  No databases found!")
    print("\nPlease run the fetch_live_databases.ipynb notebook first to download databases from the server.")

print(f"{'='*80}")

INFO:research_notebooks.brigado_v2.data_consolidator:Discovered 2 database(s)



DISCOVERED DATABASES

[1] Bot: pmm-mister-all-20260303-011107
    Database: pmm-mister-all-20260303-011107.sqlite
    Size: 43.87 MB
    Modified: 2026-03-04 21:35:30.478969
    Config Files: 5
      - brigado-binance-21-1.yml
      - brigado-binance-21-3.yml
      - brigado-binance-21-2.yml
      - brigado-binance-18-1.yml
      - brigado-binance-21-4.yml

[2] Bot: pmm-btcbrl-20260303-012028
    Database: pmm-btcbrl-20260303-012028.sqlite
    Size: 42.45 MB
    Modified: 2026-03-04 21:33:54.335917
    Config Files: 5
      - brigado-binance-btcbrl-4.yml
      - brigado-binance-btcbrl-5.yml
      - brigado-binance-btcbrl-1.yml
      - brigado-binance-btcbrl-2.yml
      - brigado-binance-btcbrl-3.yml



In [3]:
# Consolidate all databases
print("\nStarting consolidation...\n")

output_paths = consolidator.consolidate_all(force_refresh=True)

print("\n✅ Consolidation complete!")
print("\nOutput files:")
for data_type, path in output_paths.items():
    if data_type != 'metadata':
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  - {data_type}: {path.name} ({size_mb:.2f} MB)")

INFO:research_notebooks.brigado_v2.data_consolidator:================================================================================
INFO:research_notebooks.brigado_v2.data_consolidator:STARTING DATA CONSOLIDATION
INFO:research_notebooks.brigado_v2.data_consolidator:================================================================================
INFO:research_notebooks.brigado_v2.data_consolidator:Discovered 2 database(s)
INFO:research_notebooks.brigado_v2.data_consolidator:Loading data from pmm-mister-all-20260303-011107...



Starting consolidation...



INFO:research_notebooks.brigado_v2.data_consolidator:  Loaded 1380 trades, 2373 orders, 35152 executors, 5 controllers
INFO:research_notebooks.brigado_v2.data_consolidator:Loading data from pmm-btcbrl-20260303-012028...
INFO:research_notebooks.brigado_v2.data_consolidator:  Loaded 10580 trades, 12028 orders, 23075 executors, 5 controllers
INFO:research_notebooks.brigado_v2.data_consolidator:
Combining data from all sources...
INFO:research_notebooks.brigado_v2.data_consolidator:  ✓ Saved 11,960 trades to consolidated_trades.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:  ✓ Saved 14,401 orders to consolidated_orders.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:  ✓ Saved 58,227 executors to consolidated_executors.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:  ✓ Saved 10 controllers to consolidated_controllers.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:  ✓ Saved metadata to consolidation_metadata.json
INFO:research_notebooks


✅ Consolidation complete!

Output files:
  - trades: consolidated_trades.parquet (0.34 MB)
  - orders: consolidated_orders.parquet (0.56 MB)
  - executors: consolidated_executors.parquet (33.60 MB)
  - controllers: consolidated_controllers.parquet (0.02 MB)


In [4]:
# Load and inspect consolidated data
print("\nLoading consolidated data for inspection...\n")

data = consolidator.load_consolidated_data(use_latest=True)

print(f"{'='*80}")
print("CONSOLIDATED DATA SUMMARY")
print(f"{'='*80}\n")

if 'trades' in data:
    trades = data['trades']
    print(f"📊 Trades: {len(trades):,} records")
    print(f"   Date Range: {trades['timestamp'].min()} to {trades['timestamp'].max()}")
    print(f"   Unique Bots: {trades['source_bot'].nunique()}")
    print(f"   Trading Pairs: {', '.join(trades['symbol'].unique()[:5])}")
    if 'market' in trades.columns:
        print(f"   Markets: {', '.join(trades['market'].unique())}")
    print()

if 'orders' in data:
    orders = data['orders']
    print(f"📝 Orders: {len(orders):,} records")
    print(f"   Unique Bots: {orders['source_bot'].nunique()}")
    print()

if 'executors' in data:
    executors = data['executors']
    print(f"⚙️  Executors: {len(executors):,} records")
    print(f"   Unique Bots: {executors['source_bot'].nunique()}")
    if 'controller_id' in executors.columns:
        print(f"   Unique Controllers: {executors['controller_id'].nunique()}")
    print()

if 'controllers' in data:
    controllers = data['controllers']
    print(f"🎮 Controllers: {len(controllers):,} records")
    print(f"   Unique Bots: {controllers['source_bot'].nunique()}")
    if 'id' in controllers.columns:
        print(f"   Controller IDs:")
        for ctrl_id in controllers['id'].unique():
            print(f"      - {ctrl_id}")
    print()

print(f"{'='*80}")

INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 11,960 trades from consolidated_trades.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 14,401 orders from consolidated_orders.parquet



Loading consolidated data for inspection...



INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 58,227 executors from consolidated_executors.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 10 controllers from consolidated_controllers.parquet


CONSOLIDATED DATA SUMMARY

📊 Trades: 11,960 records
   Date Range: 2026-03-03 01:21:16 to 2026-03-05 00:33:11
   Unique Bots: 2
   Trading Pairs: USDT-BRL, BTC-BRL
   Markets: binance

📝 Orders: 14,401 records
   Unique Bots: 2

⚙️  Executors: 58,227 records
   Unique Bots: 2
   Unique Controllers: 10

🎮 Controllers: 10 records
   Unique Bots: 2
   Controller IDs:
      - brigado-binance-18-1
      - brigado-binance-21-4
      - brigado-binance-21-1
      - brigado-binance-21-3
      - brigado-binance-21-2
      - brigado-binance-btcbrl-1
      - brigado-binance-btcbrl-2
      - brigado-binance-btcbrl-3
      - brigado-binance-btcbrl-4
      - brigado-binance-btcbrl-5



In [5]:
# Display sample data
print("\n📋 Sample Trades (first 5 rows):\n")
if 'trades' in data and len(data['trades']) > 0:
    sample_cols = ['timestamp', 'symbol', 'trade_type', 'price', 'amount', 'source_bot']
    available_cols = [col for col in sample_cols if col in data['trades'].columns]
    display(data['trades'][available_cols].head())
else:
    print("No trade data available")

print("\n🎮 Sample Controllers:\n")
if 'controllers' in data and len(data['controllers']) > 0:
    ctrl_cols = ['id', 'source_bot']
    available_ctrl_cols = [col for col in ctrl_cols if col in data['controllers'].columns]
    display(data['controllers'][available_ctrl_cols])
else:
    print("No controller data available")


📋 Sample Trades (first 5 rows):



,timestamp,symbol,trade_type,price,amount,source_bot
0,2026-03-03 08:40:19,USDT-BRL,SELL,5.1763,149.8,pmm-mister-all-20260303-011107
1,2026-03-03 08:40:19,USDT-BRL,SELL,5.1763,149.8,pmm-mister-all-20260303-011107
2,2026-03-03 08:40:19,USDT-BRL,SELL,5.1763,149.8,pmm-mister-all-20260303-011107
3,2026-03-03 08:40:19,USDT-BRL,SELL,5.1763,149.8,pmm-mister-all-20260303-011107
4,2026-03-03 08:48:28,USDT-BRL,BUY,5.1787,598.9,pmm-mister-all-20260303-011107



🎮 Sample Controllers:



,id,source_bot
0,brigado-binance-18-1,pmm-mister-all-20260303-011107
1,brigado-binance-21-4,pmm-mister-all-20260303-011107
2,brigado-binance-21-1,pmm-mister-all-20260303-011107
3,brigado-binance-21-3,pmm-mister-all-20260303-011107
4,brigado-binance-21-2,pmm-mister-all-20260303-011107
5,brigado-binance-btcbrl-1,pmm-btcbrl-20260303-012028
6,brigado-binance-btcbrl-2,pmm-btcbrl-20260303-012028
7,brigado-binance-btcbrl-3,pmm-btcbrl-20260303-012028
8,brigado-binance-btcbrl-4,pmm-btcbrl-20260303-012028
9,brigado-binance-btcbrl-5,pmm-btcbrl-20260303-012028


In [6]:
# Per-bot statistics with controller breakdown
if 'trades' in data and len(data['trades']) > 0:
    print("\n📈 Per-Bot Trading Statistics:\n")
    
    trades = data['trades']
    trades["quote_volume"] = trades["amount"] * trades["price"]
    
    # Check available columns
    print("Available columns in trades:", trades.columns.tolist()[:20])
    print("Available columns in executors:", data['executors'].columns.tolist()[:20] if 'executors' in data else "No executors")
    print("Available columns in controllers:", data['controllers'].columns.tolist() if 'controllers' in data else "No controllers")
    print()
    
    # Try to understand the relationship
    if 'executors' in data and len(data['executors']) > 0:
        print("\nExecutor sample:")
        print(data['executors'][['controller_id', 'id']].head() if 'controller_id' in data['executors'].columns else "No controller_id")
        print()
        
    if 'config_file_path' in trades.columns:
        print("\nTrades config_file_path sample:")
        print(trades['config_file_path'].value_counts().head())
        print()
    
    # Aggregate by bot
    stats = trades.groupby('source_bot').agg({
        'timestamp': ['min', 'max', 'count'],
        'amount': 'sum',
        'quote_volume': 'sum'
    }).round(2)
    
    stats.columns = ['Start Date', 'End Date', 'Num Trades', 'Total Amount', 'Total Volume']
    
    print("Bot-level stats:")
    print(stats)
    print()
    
    # Try to map trades to controllers via config_file_path
    if 'config_file_path' in trades.columns:
        print("\n🔄 Trades by Config File Path:\n")
        for bot in trades['source_bot'].unique():
            bot_trades = trades[trades['source_bot'] == bot]
            print(f"  {bot}:")
            config_counts = bot_trades['config_file_path'].value_counts()
            for config, count in config_counts.items():
                print(f"    - {config}: {count} trades")
            print()



📈 Per-Bot Trading Statistics:

Available columns in trades: ['config_file_path', 'strategy', 'market', 'symbol', 'base_asset', 'quote_asset', 'timestamp', 'order_id', 'trade_type', 'order_type', 'price', 'amount', 'leverage', 'trade_fee', 'trade_fee_in_quote', 'exchange_trade_id', 'position', 'source_bot', 'source_db', 'quote_volume']
Available columns in executors: ['id', 'timestamp', 'type', 'close_type', 'close_timestamp', 'status', 'config', 'net_pnl_pct', 'net_pnl_quote', 'cum_fees_quote', 'filled_amount_quote', 'is_active', 'is_trading', 'custom_info', 'controller_id', 'config_parsed', 'custom_info_parsed', 'source_bot', 'source_db']
Available columns in controllers: ['id', 'controller_id', 'timestamp', 'type', 'config', 'config_parsed', 'source_bot', 'source_db']


Executor sample:
          controller_id                                            id
0  brigado-binance-21-1  6RE8ATkMbdGj2uGzUb5D1vEECxUc7x1eburPyif6uz7z
1  brigado-binance-21-4  HE1U7MugerLh1jRk67BNDBntKzaphfbYPH

In [7]:
# View consolidation metadata
metadata = consolidator.get_consolidation_info()

if metadata:
    print("\n📋 Consolidation Metadata:\n")
    print(f"Timestamp: {metadata['consolidation_datetime']}")
    print(f"Number of Databases: {metadata['num_databases']}")
    print(f"\nRecord Counts:")
    for key, value in metadata['record_counts'].items():
        print(f"  - {key}: {value:,}")
    
    print(f"\nSource Databases:")
    for db in metadata['databases']:
        print(f"  - {db['bot_name']}: {db['size_mb']:.2f} MB")
else:
    print("No consolidation metadata found")


📋 Consolidation Metadata:

Timestamp: 2026-03-04T21:45:30.691407
Number of Databases: 2

Record Counts:
  - trades: 11,960
  - orders: 14,401
  - executors: 58,227
  - controllers: 10

Source Databases:
  - pmm-mister-all-20260303-011107: 43.87 MB
  - pmm-btcbrl-20260303-012028: 42.45 MB


In [8]:
# ============================================================================
#                      GENERATE HTML REPORT
# ============================================================================

from research_notebooks.brigado_v2.html_generator import generate_consolidation_report_html
import numpy as np

print("\n" + "="*80)
print("GENERATING HTML REPORT")
print("="*80 + "\n")

# Create mapping from order_id to controller_id via executors
print("Creating trade-to-controller mapping...")
order_to_controller = {}

if 'executors' in data and len(data['executors']) > 0:
    executors_df = data['executors']
    executors_filtered = executors_df[executors_df['net_pnl_quote'] != 0]
    print(f"Filtering executors: {len(executors_df)} total -> {len(executors_filtered)} with net_pnl_quote != 0")
    
    for _, executor in executors_filtered.iterrows():
        controller_id = executor.get('controller_id')
        if controller_id and 'custom_info_parsed' in executor and isinstance(executor['custom_info_parsed'], dict):
            order_ids = executor['custom_info_parsed'].get('order_ids', [])
            if isinstance(order_ids, np.ndarray):
                order_ids = order_ids.tolist()
            elif not isinstance(order_ids, (list, tuple)):
                order_ids = list(order_ids) if order_ids else []
            for oid in order_ids:
                if oid:
                    order_to_controller[str(oid)] = controller_id

print(f"Mapped {len(order_to_controller)} orders to controllers")

# Add controller_id to trades
if 'trades' in data:
    trades_with_ctrl = data['trades'].copy()
    trades_with_ctrl['controller_id'] = trades_with_ctrl['order_id'].map(order_to_controller)
    coverage = (trades_with_ctrl['controller_id'].notna().sum() / len(trades_with_ctrl)) * 100
    print(f"Controller coverage: {coverage:.1f}% of trades mapped")
    
    print("\nCoverage by bot:")
    for bot in trades_with_ctrl['source_bot'].unique():
        bot_trades = trades_with_ctrl[trades_with_ctrl['source_bot'] == bot]
        bot_mapped = bot_trades['controller_id'].notna().sum()
        bot_total = len(bot_trades)
        bot_pct = (bot_mapped / bot_total * 100) if bot_total > 0 else 0
        print(f"  {bot}: {bot_mapped}/{bot_total} ({bot_pct:.1f}%)")
else:
    trades_with_ctrl = data['trades'].copy()

# Generate HTML report
if metadata and len(trades_with_ctrl) > 0:
    html_path = file_manager.data_sources_dir / "consolidation_report.html"
    generate_consolidation_report_html(
        output_path=html_path,
        metadata=metadata,
        databases=databases,
        trades_with_ctrl=trades_with_ctrl,
        controllers_data=data['controllers'],
        output_paths=output_paths
    )
    
    print(f"\n✅ HTML report generated successfully!")
    print(f"\nReport saved to:")
    print(f"  - {html_path}")
    print(f"\n💡 Open the report in your browser to view the consolidation summary.")
else:
    print("\n⚠️  Cannot generate HTML report - missing data")

print("="*80)


GENERATING HTML REPORT

Creating trade-to-controller mapping...
Filtering executors: 58227 total -> 4799 with net_pnl_quote != 0
Mapped 9598 orders to controllers
Controller coverage: 92.7% of trades mapped

Coverage by bot:
  pmm-mister-all-20260303-011107: 1184/1380 (85.8%)
  pmm-btcbrl-20260303-012028: 9904/10580 (93.6%)

✅ HTML report generated successfully!

Report saved to:
  - /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/data_sources/consolidation_report.html

💡 Open the report in your browser to view the consolidation summary.


## Next Steps

Now that you have consolidated the data, you can:

1. **Run Performance Analysis** - Use `performance.ipynb` (updated version) to analyze the consolidated data
2. **Query Consolidated Data** - Load the parquet files directly for custom analysis:
   ```python
   trades = pd.read_parquet('data/data_sources/consolidated_trades_latest.parquet')
   ```
3. **Filter by Bot** - Analyze specific bots:
   ```python
   bot_trades = trades[trades['source_bot'] == 'pmm-mister-all-20260303-011107']
   ```
4. **Filter by Controller** - Analyze specific strategies (after enrichment):
   ```python
   # Will be available after running the enrichment step
   ```

## Refreshing Data

To refresh the consolidated data:
1. Run `fetch_live_databases.ipynb` to get latest databases from server
2. Re-run this notebook to rebuild consolidated files

## Data Format

The consolidated data is stored in Parquet format for:
- **Fast querying** - 10-100x faster than CSV
- **Compression** - Smaller file sizes
- **Type preservation** - Maintains data types (timestamps, floats, etc.)
- **Columnar storage** - Efficient for analytics